In [6]:
import pandas as pd
import pyodbc

In [2]:
conn = pyodbc.connect(
    'DRIVER = {SQL Server};'
    'SERVER = localhost\\SQLEXPRESS;'
    'DATABASE = MarketingAnalytics;'
    'Trusted_Connection = yes;'
)

InterfaceError: ('IM002', '[IM002] [Microsoft][ODBC Driver Manager] Data source name not found and no default driver specified (0) (SQLDriverConnect)')

In [3]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [4]:
conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=MarketingAnalytics;'
    'Trusted_Connection=yes;'
)

In [5]:
query = "SELECT * FROM customer_reviews"
df = pd.read_sql(query, conn)
df.head()

C:\Users\Nainesh19072005\AppData\Local\Temp\ipykernel_12212\1165095198.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [6]:
pip install textblob

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [7]:
from textblob import TextBlob

In [8]:
def get_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    else:
        return "Neutral"

In [9]:
df['Sentiment'] = df['ReviewText'].apply(get_sentiment)

In [10]:
df[['ReviewText', 'Sentiment']].head(20)

,ReviewText,Sentiment
0,"Average experience, nothing special.",Positive
1,The quality is top-notch.,Positive
2,Five stars for the quick delivery.,Positive
3,"Good quality, but could be cheaper.",Positive
4,"Average experience, nothing special.",Positive
5,Customer support was very helpful.,Positive
6,"Average experience, nothing special.",Positive
7,The quality is top-notch.,Positive
8,"I love this product, will buy again!",Positive
9,"Excellent product, highly recommend!",Positive


In [11]:
df.shape

(1363, 7)

In [12]:
df.to_csv("sentiment_output.csv", index=False)

In [1]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Nainesh19072005\AppData\Roaming\nltk_data...


True

In [3]:
sia = SentimentIntensityAnalyzer()

In [4]:
def calc_sentiment(review):
    sentiment = sia.polarity_scores(str(review))
    return sentiment['compound']

In [8]:
df['SentimentScore'] = df['ReviewText'].apply(calc_sentiment)

NameError: name 'df' is not defined

In [9]:
import pandas as pd

df = pd.read_csv("sentiment_output.csv")
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,Sentiment
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",Positive
1,2,80,19,2024-12-25,5,The quality is top-notch.,Positive
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,Positive
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",Positive
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",Positive


In [10]:
df['SentimentScore'] = df['ReviewText'].apply(calc_sentiment)

In [11]:
df[['ReviewText','SentimentScore']].head()

,ReviewText,SentimentScore
0,"Average experience, nothing special.",-0.3089
1,The quality is top-notch.,0.0000
2,Five stars for the quick delivery.,0.0000
3,"Good quality, but could be cheaper.",0.2382
4,"Average experience, nothing special.",-0.3089


In [12]:
def categorize_sentiment(score, rating):
    if score > 0.05:
        if rating >= 4:
            return 'Positive'
        elif rating == 3:
            return 'Mixed Positive'
        else:
            return 'Mixed Negative'

    elif score < -0.05:
        if rating <= 2:
            return 'Negative'
        elif rating == 3:
            return 'Mixed Negative'
        else:
            return 'Mixed Positive'

    else:
        if rating >= 4:
            return 'Positive'
        elif rating <= 2:
            return 'Negative'
        else:
            return 'Neutral'

In [13]:
df['SentimentCategory'] = df.apply(lambda row: categorize_sentiment(row['SentimentScore'], row['Rating']), axis = 1)

In [14]:
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'
    else:
        return '-1.0 to -0.5'

In [15]:
df['SentimentBucket'] = df['SentimentScore'].apply(sentiment_bucket)

In [19]:
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,Sentiment,SentimentScore,SentimentCategory,SentimentBucket
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",Positive,-0.3089,Mixed Negative,-0.49 to 0.0
1,2,80,19,2024-12-25,5,The quality is top-notch.,Positive,0.0000,Positive,0.0 to 0.49
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,Positive,0.0000,Positive,0.0 to 0.49
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",Positive,0.2382,Mixed Positive,0.0 to 0.49
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",Positive,-0.3089,Mixed Negative,-0.49 to 0.0


In [17]:
df.to_csv("fact_customer_reviews_with_sentiment.csv", index=False)